<figure>
  <IMG SRC="https://raw.githubusercontent.com/fhswf/datasets/refs/heads/main/images/fh-swf-logo.svg" WIDTH=250 ALIGN="right">
</figure>

# Programmierung für KI
### Winterersemester 2026/27
Prof. Dr. Heiner Giefers

In [ ]:
%load_ext tutormagic

<figure>
  <IMG SRC="https://raw.githubusercontent.com/fhswf/datasets/refs/heads/main/images/fh-swf-logo.svg" WIDTH=250 ALIGN="right">
</figure>

# Programmierung für KI
### Wintersemester 2026/27
Prof. Dr. Heiner Giefers

# Objektorientierung

![](O1.png)

![](O2.png)

## *Alles in Python ist ein Objekt!*


*Alles* = alles, was Sie mit einem Bezeichner versehen können

Instanzen von elementaren und zusammengesetzten Datentypen, Funktionen/Methoden und Klassen sind Objekte.


## Objekte kennen Sie schon

- Auch eingebaute Datentypen wie `list` sind Klassen, ihre Werte sind Objekte
- `append` oder `sort` sind *Methoden* der Klasse `list`
- Die eckigen Klammern sind nur eine Abkürzung für den Aufruf der Methode `__getitem__`

In [ ]:
stuff = list()
stuff.append('foo')
stuff.append('bar')
stuff.sort()
print(stuff[0])
print(stuff.__getitem__(0))
print(list.__getitem__(stuff, 0))

### Begriffe

- **Klasse:** Ein *Template* für ein Objekt
- **Objekt:** Die *Instanz* einer Klasse
- **Methode:** Eine *Fähigkeit* einer Klasse
- **Attribut:** Ein *Datum* (im Sinne von *Daten*) in einer Klasse oder einem Objekt


In [ ]:
%%tutor --lang python3 --cumulative --heapPrimitives

class Complex: 
    def __init__(self, realpart, imagpart):
        self.r = realpart
        self.i = imagpart 

x = Complex(1, -1)
x.r, x.i

[Python Tutor](https://pythontutor.com/visualize.html#code=class%20Complex%3A%0A%20%20%20%20def%20__init__%28self,%20realpart,%20imagpart%29%3A%0A%20%20%20%20%20%20%20%20self.r%20%3D%20realpart%0A%20%20%20%20%20%20%20%20self.i%20%3D%20imagpart%0A%0Ax%20%3D%20Complex%281,%20-1%29%0Ax.r,%20x.i&cumulative=false&curInstr=0&heapPrimitives=true&mode=display&origin=opt-frontend.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false)

## Beispiel: Konto

- Wir entwickeln heute schrittweise eine Klasse `Konto`
- An ihr zeigen wir alle Konzepte der Objektorientierung

In [ ]:
class Konto:
    def __init__(self, inhaber, saldo=0):
        self.inhaber = inhaber
        self.saldo = saldo

    def einzahlen(self, betrag):
        self.saldo += betrag

k = Konto("Ada")
k.einzahlen(100)
print(k.inhaber, k.saldo)

In [ ]:
print("Konto ist vom Typ", type(Konto))
print("k ist vom Typ", type(k))

(`type` ist eine *Metaklasse* und selbst ein Objekt dieser Klasse.)

In [ ]:
Konto.bank = "FH-Bank"
k.bank

## Attribute

- ... speichern *Daten*, die zu einer Klasse bzw. einem Objekt gehören
- Klassenattribute gehören zu einer Klasse, Instanzattribute zu einem einzelnen Objekt
- Klassen und Objekte legen Attribute in Dictionaries ab (`__dict__`)
- Greift man auf ein Instanzattribut (lesend) zu, wird zuerst im Dictionary des Objekts nachgeschaut. Gibt es dort das Attribut nicht, wird auch im `__dict__` der Klasse (und ggf. ihrer Basisklassen) geschaut.
- Instanzattribute *überdecken* Klassenattribute

In [ ]:
class Konto:
    zinssatz = 0.01                  # Klassenattribut

    def __init__(self, inhaber, saldo=0):
        self.inhaber = inhaber       # Instanzattribute
        self.saldo = saldo

ada = Konto("Ada")
bob = Konto("Bob")
bob.zinssatz = 0.05                  # überdeckt das Klassenattribut nur für bob
print(ada.zinssatz, bob.zinssatz, Konto.zinssatz)
print(ada.__dict__)
print(bob.__dict__)

## Klassenattribute

- Ein Klassenattribut existiert nur *einmal* und wird von allen Objekten *geteilt*
- Bei unveränderlichen Werten (Zahlen, Strings) fällt das kaum auf
- Bei veränderlichen Werten (Listen, Dictionaries) ändern alle Objekte *dasselbe* Objekt

In [ ]:
class Konto:
    buchungen = []                   # Achtung: gehört der Klasse!

    def __init__(self, inhaber):
        self.inhaber = inhaber

    def einzahlen(self, betrag):
        self.buchungen.append(betrag)

ada = Konto("Ada")
bob = Konto("Bob")
ada.einzahlen(100)
print(bob.buchungen)

- Die gleiche Falle gibt es bei veränderlichen *Default-Argumenten*
- Der Default-Wert wird nur *einmal* erzeugt, nämlich wenn die `def`-Zeile ausgeführt wird

In [ ]:
class Konto:
    def __init__(self, inhaber, buchungen=[]):   # Achtung!
        self.inhaber = inhaber
        self.buchungen = buchungen

ada = Konto("Ada")
bob = Konto("Bob")
ada.buchungen.append(100)
print(bob.buchungen)

- Lösung: veränderliche Attribute im Konstruktor anlegen (`self.buchungen = []`)
- Bei Default-Argumenten `None` verwenden und die Liste erst im Konstruktor erzeugen

In [ ]:
class Konto:
    def __init__(self, inhaber, buchungen=None):
        self.inhaber = inhaber
        if buchungen is None:
            self.buchungen = []
        else:
            self.buchungen = buchungen

ada = Konto("Ada")
bob = Konto("Bob")
ada.buchungen.append(100)
print(bob.buchungen)

## Konstruktor und Destruktor

- Ein *Konstruktor* ist eine spezielle Methode (`__init__`), die beim Erzeugen eines Objekts aufgerufen wird
- Üblicherweise initialisiert der Konstruktor die (wichtigsten) Attribute eines Objekts
- Der *Destruktor* (`__del__`) wird aufgerufen, wenn die letzte Referenz auf ein Objekt "gelöscht" wird (in CPython sofort, garantiert ist der Zeitpunkt aber nicht)
- Destruktoren werden nur selten benötigt

In [ ]:
class Konto:
    anzahl = 0

    def __init__(self, inhaber):
        self.inhaber = inhaber
        Konto.anzahl += 1

    def __del__(self):
        Konto.anzahl -= 1

k1 = Konto("Ada")
k2 = Konto("Bob")
k3 = Konto("Cem")
print(f"Es gibt {Konto.anzahl} Konten")
k2 = 42
print(f"Es gibt nun {Konto.anzahl} Konten")
del k1, k2, k3

In [ ]:
class A():
    ''' Meine Klasse A'''
    alle_As = []
    
    def __init__(self, value):
        self.value = value
        A.alle_As.append(self)

    def __del__(self):
        self.alle_As.remove(self)
        
a0 = A(0)
a1 = A(1)
a2 = A(2)
a3 = A(3)
a2 = 42

for a in A.alle_As:
    print(a.value)
    
del a0, a1, a2, a3

[Python Tutor](https://pythontutor.com/visualize.html#code=class%20A%28%29%3A%0A%20%20%20%20'''%20Meine%20Klasse%20A'''%0A%20%20%20%20the_as%20%3D%20%5B%5D%0A%20%20%20%20%0A%20%20%20%20def%20__init__%28self,%20value%29%3A%0A%20%20%20%20%20%20%20%20self.value%20%3D%20value%0A%20%20%20%20%20%20%20%20A.the_as.append%28self%29%0A%0A%20%20%20%20def%20__del__%28self%29%3A%0A%20%20%20%20%20%20%20%20A.the_as.remove%28self%29%0A%20%20%20%20%20%20%20%20%0Aa0%20%3D%20A%280%29%0Aa1%20%3D%20A%281%29%0Aa2%20%3D%20A%282%29%0Aa3%20%3D%20A%283%29%0Aa2%20%3D%2042%0A%0Afor%20a%20in%20A.the_as%3A%0A%20%20%20%20print%20%28a.value%29%0A%20%20%20%20%0Adel%20a0,%20a1,%20a2,%20a3&cumulative=false&curInstr=0&heapPrimitives=true&mode=display&origin=opt-frontend.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false)

- Um die Objekte im Speicher kümmert sich der **Garbage Collector**
- Objekte werden aus dem Speicher gelöscht, wenn sie nicht mehr *erreichbar* sind, z.B. weil es keine *Referenz* mehr auf das Objekt gibt

## Methoden

- ...sind Funktionen, die *etwas mit Objekten einer Klasse tun*
- Jede (normale) Methode verwendet mindestens ein Objekt der Klasse
- Eine Methode ist im *Namensraum* der Klasse definiert
- Eine Methode kann mit `Klassenname.Methodenname(Objektreferenz)` aufgerufen werden

- Besser: Aufruf mit `Objektreferenz.Methodenname`


In [ ]:
class Konto:
    def __init__(self, inhaber, saldo=0):
        self.inhaber = inhaber
        self.saldo = saldo

    def einzahlen(self, betrag):
        self.saldo += betrag
        print(f"{self.inhaber}: neuer Saldo {self.saldo}")

k = Konto("Ada")
Konto.einzahlen(k, 100)
k.einzahlen(50)

- Der erste Parameter einer Methode bezeichnet immer **das Objekt, auf das die Methode angewendet wird**
- Diesen Parameter nennt man auch *Selbstreferenz*
- Per Konvention wird diese Selbstreferenz `self` genannt
- Beim Aufruf einer Methode über die Punkt-Notation wird das Objekt ("vor dem Punkt") als Argument für `self` eingesetzt

- `k.einzahlen` ist eine *gebundene Methode*: Das Objekt für `self` ist bereits festgelegt
- Gebundene Methoden kann man wie jedes andere Objekt speichern und später aufrufen

In [ ]:
print(Konto.einzahlen)
print(k.einzahlen)
buche = k.einzahlen
buche(10)

- Es gibt auch Funktionen, die zwar zu einer Klasse gehören, aber *nichts* mit einem bestimmten Objekt der Klasse *tun*
- Solche Funktionen heißen *statische Methoden*
- Um sie auch über Objekte aufrufen zu können, muss man sie mit dem *Dekorator* `@staticmethod` versehen

- *Klassenmethoden* (Dekorator `@classmethod`) bekommen statt des Objekts die *Klasse* übergeben (`cls`)
- Typische Anwendung: *alternative Konstruktoren*

In [ ]:
class Konto:
    def __init__(self, inhaber, saldo=0):
        self.inhaber = inhaber
        self.saldo = saldo

    @staticmethod
    def ist_gueltiger_betrag(betrag):
        return isinstance(betrag, (int, float)) and betrag > 0

    @classmethod
    def aus_dict(cls, daten):
        return cls(daten["inhaber"], daten.get("saldo", 0))

print(Konto.ist_gueltiger_betrag(-5))
k = Konto.aus_dict({"inhaber": "Ada", "saldo": 100})
print(k.inhaber, k.saldo)

## Vererbung

- Zentrales Konzept in der objektorientierten Programmierung
- (Neue) Klassen können von bestehenden Klassen **abgeleitet** werden und **erben** dadurch alle Methoden und Attribute ihrer **Elternklasse** (auch *Basisklasse* genannt)

In [ ]:
class Sparkonto(Konto):
    pass

s = Sparkonto("Ada", 100)
print(s.inhaber, s.saldo, isinstance(s, Konto))
print(type(Sparkonto.aus_dict({"inhaber": "Bob"})))   # cls ist hier Sparkonto

## Eigene Exceptions

- Auch Exceptions sind Klassen und bilden eine Vererbungshierarchie
- Eigene Exceptions leitet man von einer passenden vorhandenen Exception ab
- `except` fängt auch alle *abgeleiteten* Exceptions

In [ ]:
class DeckungsFehler(ValueError):
    pass

try:
    raise DeckungsFehler("Deckung nicht ausreichend")
except ValueError as e:
    print("Fehler:", type(e).__name__, "-", e)

print(DeckungsFehler.__mro__)

## Überschreiben und `super()`

- Die abgeleitete Klasse kann für ihren eigenen Zweck angepasst und erweitert werden
- Es können neue Attribute und Methoden eingeführt werden oder auch bestehende Attribute und Methoden von der Basisklasse überschrieben werden
- Ist eine aufgerufene Methode in der Klasse eines Objekts nicht vorhanden, wird nach der Methode in den Basisklassen gesucht
- Die erste *passende* Methode wird aufgerufen ($\rightarrow$ *Method Resolution Order*, MRO)
- Es kann mehrere Basisklassen geben ($\rightarrow$ *Mehrfachvererbung*)
- Um explizit auf Attribute und Methoden der Basisklasse zuzugreifen, kann man die Funktion `super()` verwenden
- Mit `super().__init__()` kann der übergeordnete Konstruktor aufgerufen werden
- Sinnvoll, um doppelten Code zu vermeiden

In [ ]:
class Konto:
    def __init__(self, inhaber, saldo=0):
        self.inhaber = inhaber
        self.saldo = saldo

    def abheben(self, betrag):
        if betrag > self.saldo:
            raise DeckungsFehler("Deckung nicht ausreichend")
        self.saldo -= betrag

class Girokonto(Konto):
    def __init__(self, inhaber, saldo=0, dispo=500):
        super().__init__(inhaber, saldo)
        self.dispo = dispo

    def abheben(self, betrag):                    # überschreibt Konto.abheben
        if betrag > self.saldo + self.dispo:
            raise DeckungsFehler("Dispo überschritten")
        self.saldo -= betrag

g = Girokonto("Ada", 100)
g.abheben(300)
print(g.saldo)

## Polymorphie

- Objekte unterschiedlicher Klassen reagieren auf denselben Methodenaufruf *unterschiedlich*
- Der Aufrufer muss die konkrete Klasse nicht kennen

In [ ]:
def alle_abheben(konten, betrag):
    for k in konten:
        try:
            k.abheben(betrag)
            print(f"{k.inhaber}: {k.saldo}")
        except DeckungsFehler as e:
            print(f"{k.inhaber}: {e}")

konten = [Konto("Ada", 100), Girokonto("Bob", 100), Konto("Cem", 500)]
alle_abheben(konten, 200)

- In Python reicht es, dass ein Objekt die passende Methode *hat* (*Duck Typing*)

In [ ]:
class Spardose:                     # erbt nicht von Konto!
    def __init__(self):
        self.inhaber = "Spardose"
        self.saldo = 20

    def abheben(self, betrag):
        self.saldo -= betrag

alle_abheben([Spardose()], 5)

## Abstrakte Basisklassen

- Manche Klassen sollen nur eine *Schnittstelle* vorgeben, aber selbst nicht instanziiert werden
- Das Modul `abc` erzwingt, dass abgeleitete Klassen bestimmte Methoden implementieren

In [ ]:
from abc import ABC, abstractmethod

class Konto(ABC):
    def __init__(self, inhaber, saldo=0):
        self.inhaber = inhaber
        self.saldo = saldo

    @abstractmethod
    def abheben(self, betrag):
        pass

In [ ]:
try:
    k = Konto("Bob")
except TypeError as e:
    print(e)

In [ ]:
class Sparkonto(Konto):
    def abheben(self, betrag):
        if betrag > self.saldo:
            raise DeckungsFehler("Deckung nicht ausreichend")
        self.saldo -= betrag

s = Sparkonto("Ada", 100)
s.abheben(40)
print(s.saldo)

- Im Unterschied zu Java (aber ähnlich zu C++) ist in Python **Mehrfachvererbung** möglich
- Es können bei der Klassen-Definition mehrere Basisklassen angegeben werden
- Dadurch ergeben sich Fragestellungen, vor allem bezüglich der Eindeutigkeit von Referenzen
- Mit der *Method Resolution Order* besitzt Python ein festgelegtes Verfahren, um übergeordneten Klassen eine definierte Rangfolge zu geben

In [ ]:
class O():
    def f(self): print("f() aus O")
class A(O):
    pass
class B(O):
    def f(self): print("f() aus B")
class C(A,B):
    pass

c = C()
c.f()
print([k.__name__ for k in C.__mro__])

In [ ]:
class A():
    def __init__(self):
        print("ich bin ein A")
class B():
    def __init__(self):
        print("ich bin ein B")

class C(A,B):
    def __init__(self):
        A.__init__(self)
        print("und")
        B.__init__(self)
c=C()

## Magic Methods und Magic Attributes
- In Python gibt es eine ganze Reihe von speziellen Attributen und Methoden mit einer vordefinierten Funktion
- Der Interpreter verwendet bei bestimmten Operationen die entsprechenden Methoden (oder Attribute)
- Man kann in eigenen Klassen die Funktionsweise (bei Methoden) oder die Werte (bei Attributen) *überschreiben* bzw. *überdecken*
- Beispiel: Wenn ein Objekt erzeugt werden soll, wird immer die Methode `__new__` und dann `__init__` aufgerufen
- Alle *Magic Methods* und *Magic Attributes* sind in doppelten Unterstrichen `__` eingefasst
- Es gibt weitere *Magic Methods*, mit denen Sie Operatoren *überladen* können ($\rightarrow$ Praktikumsaufgabe von heute)

In [ ]:
class A():
    pass

print(dir(A))
print("Das sind nicht alle...")

## `__str__` und `__repr__`

- `__str__`: lesbare Darstellung für Anwender (`print`, `str()`)
- `__repr__`: eindeutige Darstellung für Entwickler (Konsole, Listen, Debugger)
- Fehlt `__str__`, wird `__repr__` verwendet

In [ ]:
class Konto:
    def __init__(self, inhaber):
        self.inhaber = inhaber
        self.buchungen = []

    def einzahlen(self, betrag):
        self.buchungen.append(betrag)

    def __str__(self):
        return f"Konto von {self.inhaber}: {sum(self.buchungen):.2f} €"

    def __repr__(self):
        return f"Konto({self.inhaber!r})"

k = Konto("Ada")
k.einzahlen(100)
print(k)
k

## Eigene Objekte wie eingebaute Typen

- `len(k)` ruft `k.__len__()` auf, `k[0]` ruft `k.__getitem__(0)` auf
- `for` funktioniert hier sogar ohne `__iter__`, weil `__getitem__` vorhanden ist

In [ ]:
class Konto:
    def __init__(self, inhaber):
        self.inhaber = inhaber
        self.buchungen = []

    def einzahlen(self, betrag):
        self.buchungen.append(betrag)

    def __repr__(self):
        return f"{type(self).__name__}({self.inhaber!r})"

    def __len__(self):
        return len(self.buchungen)

    def __getitem__(self, i):
        return self.buchungen[i]

ada = Konto("Ada"); ada.einzahlen(100); ada.einzahlen(20)
print(len(ada), ada[0], ada[-1])
print([b for b in ada])

- `==` nutzt `__eq__`, `<` (und damit `sorted`) nutzt `__lt__`

In [ ]:
class VergleichbaresKonto(Konto):
    def __eq__(self, other):
        return self.inhaber == other.inhaber

    def __lt__(self, other):
        return sum(self.buchungen) < sum(other.buchungen)

ada = VergleichbaresKonto("Ada"); ada.einzahlen(100); ada.einzahlen(20)
bob = VergleichbaresKonto("Bob"); bob.einzahlen(50)
print(sorted([ada, bob]))
print(ada == VergleichbaresKonto("Ada"))

- Achtung: Wer `__eq__` definiert, macht Objekte *unhashbar* (`__hash__` wird auf `None` gesetzt)
- Solche Objekte können nicht in Sets oder als Schlüssel in Dictionaries verwendet werden

In [ ]:
try:
    konten = {ada, bob}
except TypeError as e:
    print(e)

## Aufrufbare Objekte: `__call__`

- Definiert eine Klasse `__call__`, kann man ihre Objekte *wie Funktionen aufrufen*
- `objekt(x)` ist eine Abkürzung für `objekt.__call__(x)`
- So kann sich eine "Funktion" Werte merken (hier: den Zinssatz)
- Bezug zur *funktionalen Programmierung*: Funktionen sind in Python selbst Objekte (mit einer Methode `__call__`) und können wie Daten übergeben werden
- Aufrufbare Objekte sind damit eine Alternative zu *Closures*, also Funktionen, die sich Werte aus ihrer Umgebung merken

In [ ]:
class Zinsrechner:
    def __init__(self, zinssatz):
        self.zinssatz = zinssatz

    def __call__(self, betrag):
        return betrag * (1 + self.zinssatz)

zins_2 = Zinsrechner(0.02)
print(zins_2(1000))
print(callable(zins_2))

### Beispiel
- Schreibe eine Klasse Punkt sowie eine Klasse Punkt3D für Punkte in der Ebene bzw. im Raum
- Punkte sollen ein Label bekommen können
- Punkte sollen sich sinnvoll ausgeben (drucken) lassen

In [ ]:
class Punkt():
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
class Punkt3D(Punkt):
    pass

p1 = Punkt3D(2.0, -4.1, 3.3)
p1.set_label("Erster Punkt")
print(p1)

In [ ]:
class Punkt():
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.label = None
    def set_label(self, label):
        self.label = label

class Punkt3D(Punkt):
    def __init__(self, x, y, z):
        super().__init__(x,y)
        self.z = z
    def __str__(self):
        res = ""
        if self.label:
            res += self.label + ": "
        res += f"({self.x}, {self.y}, {self.z})"
        return res

p1 = Punkt3D(2.0, -4.1, 3.3)
p1.set_label("Erster Punkt")
print(p1)

## Zugriffsregeln

- Wenn man ein Objekt (einer Klasse) verwendet, muss man i.d.R. nicht *alles über die Klasse wissen*
- Bestimmte Methoden und Attribute werden nur *innerhalb* der Klasse verwendet, aber nicht *von außen*
- Viele objektorientierte Sprachen verwenden **Zugriffsmodifikatoren** (z.B. *public* und *private*), um die **Sichtbarkeit** von Methoden und Attributen einzugrenzen

- In Python gibt es solche Zugriffsmodifikatoren *nicht direkt*
- Über Namenskonventionen kann man ausdrücken, ob ein Attribut geschützt ist

- **Einfacher Unterstrich (`_x`):** Das Attribut `_x` *sollte* extern nicht direkt verwendet werden
- **Doppelter Unterstrich (`__x`):** Das Attribut `__x` *kann* extern nicht (ohne Weiteres) direkt verwendet werden

In [ ]:
class A():
    def __init__(self):
        self.x = 100
        self._x = 200
        self.__x = 300
    def __str__(self):
        return f"x = {self.x}, _x = {self._x}, __x = {self.__x}"

a = A()
print(a)
print(a.x)
print(a._x)
#a.__dict__
#print(a._A__x)

## Properties (<s>Getter und Setter</s>)

- Zugriff sieht aus wie bei einem Attribut, ruft aber Methoden auf
- Nützlich für Validierung und berechnete Werte

In [ ]:
class Konto:
    def __init__(self, inhaber):
        self.inhaber = inhaber           # nutzt bereits set_inhaber

    def get_inhaber(self):
        print("Lese _inhaber")
        return self._inhaber

    def set_inhaber(self, name):   
        if not name: raise ValueError("Inhaber darf nicht leer sein");
        else: print("Setze _inhaber");
        self._inhaber = name

    inhaber = property(get_inhaber, set_inhaber)

k = Konto("Ada")
k.inhaber = "Bob"
print(k.inhaber)
#k.inhaber = ""

- Kürzer und heute üblich: Properties mit dem *Dekorator* `@property`

In [ ]:
class Konto:
    def __init__(self, inhaber, saldo=0):
        self.inhaber = inhaber
        self._saldo = saldo

    @property
    def saldo(self):
        return self._saldo

k = Konto("Ada", 100)
print(k.saldo)

- Mit `@<name>.setter` lässt sich auch das *Schreiben* steuern, z.B. zur *Validierung*

In [ ]:
class Girokonto(Konto):
    def __init__(self, inhaber, saldo=0, dispo=0):
        super().__init__(inhaber, saldo)
        self.dispo = dispo           # nutzt bereits den Setter

    @property
    def dispo(self):
        return self._dispo

    @dispo.setter
    def dispo(self, wert):
        if wert < 0:
            raise ValueError("Dispo darf nicht negativ sein")
        self._dispo = wert

g = Girokonto("Ada", dispo=500)
print(g.dispo)
try:
    g.dispo = -100
except ValueError as e:
    print("Fehler:", e)

## Ausblick: Dataclasses

- Viele Klassen speichern vor allem Daten
- Der Dekorator `@dataclass` erzeugt `__init__`, `__repr__` und `__eq__` automatisch (benötigt *Typannotationen*)
- `field(default_factory=list)`: Für jedes neue Objekt wird `list()` aufgerufen, jedes Konto bekommt also seine *eigene* leere Liste

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Buchung:
    betrag: float
    text: str = ""

@dataclass
class Konto:
    inhaber: str
    buchungen: list = field(default_factory=list)

k = Konto("Ada")
k.buchungen.append(Buchung(100, "Gehalt"))
print(k)
print(Buchung(5) == Buchung(5))

## Zum 6. Termin

- **NumPy**: Bibliothek für numerische Operationen mit Python

## Beispiel

- Schreibe eine Klasse Punkt, die jeden neu erzeugten Punkt in einer Liste speichert
- Die Punkte sollen in ein *Koordinatensystem* geplottet werden können

In [ ]:
#todo

In [ ]:
p0 = Punkt(0,0)
p1 = Punkt(3,4)
p2 = Punkt(5,2)
p4 = Punkt(6,3) 
Punkt.plot()

In [ ]:
import matplotlib.pyplot as plt
from itertools import cycle
cycol = cycle('bgrcmk')

class Punkt():
        
    alle_punkte = []

    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.alle_punkte.append(self)
    
    @staticmethod
    def plot():
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        plt.xlim((-10,10))
        plt.ylim((-6,6))
    
        if len(Punkt.alle_punkte)>0:
            punkte = Punkt.alle_punkte  + [Punkt.alle_punkte[0]]
            xx = [p.x for p in punkte]
            yy = [p.y for p in punkte]  
            ax.plot(xx, yy, '-', c=next(cycol))
            #ax.scatter(xx, yy)
            
p0 = Punkt(0,0)
p1 = Punkt(3,4)
p2 = Punkt(5,2)
p4 = Punkt(6,3)
Punkt.plot()

## Beispiel

- Schreibe eine Klasse Punkt sowie eine Klasse Punkt3D für Punkte in der Ebene bzw. im Raum
- Punkte sollen ein Label bekommen können
- Punkte sollen sich sinnvoll ausgeben (drucken) lassen
- Die Punkte sollen sich verschieben lassen

In [ ]:
class Punkt():
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.label = None
    def set_label(self, label):
        self.label = label

class Punkt3D(Punkt):
    def __init__(self, x, y, z):
        super().__init__(x,y)
        self.z = z
    def __str__(self):
        res = ""
        if self.label:
            res += self.label + ": "
        res += f"({self.x:.3}, {self.y:.3}, {self.z:.3})"
        return res

p1 = Punkt3D(2.0, -4.1, 3.3)
p1.set_label("Erster Punkt")
print(p1)
#p1.verschiebe((1.0,5.0,-4.0))
#print(p1)

In [ ]:
class Punkt():
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.label = None
    def set_label(self, label):
        self.label = label
    def verschiebe(self, coord):
        if len(coord)==2:
            self.x += coord[0]
            self.y += coord[1]

class Punkt3D(Punkt):
    def __init__(self, x, y, z):
        super().__init__(x,y)
        self.z = z
    def __str__(self):
        res = ""
        if self.label:
            res += self.label + ": "
        res += f"({self.x:.3}, {self.y:.3}, {self.z:.3})"
        return res
    def verschiebe(self, coord):
        if len(coord)==3:
            self.x += coord[0]
            self.y += coord[1]
            self.z += coord[2]

p1 = Punkt3D(2.0, -4.1, 3.3)
p1.set_label("Erster Punkt")
print(p1)
p1.verschiebe((1.0,5.0,-4.0))
print(p1)
del p1